In [ ]:
# import os
# import re
# import numpy as np
# import pandas as pd
# from typing import Optional
# import json 
# import pickle 

# import torch
# import torch.nn as nn
# from torch.utils.data import Dataset, DataLoader
# from transformers import BertTokenizer, BertModel
# from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import classification_report, confusion_matrix
# from sklearn.model_selection import GroupShuffleSplit
# import matplotlib.pyplot as plt
# import seaborn as sns

# # ══════════════════════════════════════════════════════════════════════════════
# # UTILITY FUNCTIONS
# # ══════════════════════════════════════════════════════════════════════════════

# def clean_currency(value) -> float:
#     """Helper to convert scraped strings like '$11,400,000' to a float."""
#     if pd.isna(value) or value == "N/A":
#         return np.nan
#     clean_str = re.sub(r'[^\d.]', '', str(value))
#     try:
#         return float(clean_str)
#     except ValueError:
#         return np.nan

# def parse_movie_name(filename: str) -> str:
#     """Parses movie key from filename (e.g., 'reviews_imdb_parasite.csv' -> 'parasite')"""
#     name = os.path.splitext(filename)[0].lower()
#     known_platforms = {"imdb", "rt", "letter", "letterboxd", "metacritic", "rottentomatoes"}
#     for platform in known_platforms:
#         name = re.sub(rf"_?{platform}_?", "_", name)
#     name = re.sub(r"_?reviews_?$", "", name)
#     name = re.sub(r"_+", "_", name).strip("_")
#     return name

# def parse_user_rating(val) -> float:
#     """Normalizes various rating formats to a 0-10 float scale."""
#     if pd.isna(val): return np.nan
#     s = str(val).strip()
#     pct = re.match(r"^([\d.]+)%$", s)
#     if pct: return float(pct.group(1)) / 10.0
#     frac = re.match(r"^([\d.]+)\s*/\s*([\d.]+)$", s)
#     if frac:
#         num, denom = float(frac.group(1)), float(frac.group(2))
#         return (num / denom) * 10.0 if denom != 0 else np.nan
#     try:
#         return float(s)
#     except ValueError:
#         return np.nan

# # ══════════════════════════════════════════════════════════════════════════════
# # DATA LOADING & LABELING
# # ══════════════════════════════════════════════════════════════════════════════

# def load_all_csvs(folder_path: str) -> pd.DataFrame:
#     metadata_path = os.path.join(folder_path, "movie_metadata.csv")
#     if not os.path.exists(metadata_path):
#         raise FileNotFoundError(f"Missing metadata file at: {metadata_path}")
    
#     meta_df = pd.read_csv(metadata_path)
#     # Create lookup for financial and metadata info
#     financial_lookup = meta_df.set_index('movie_key')[['budget', 'gross_us', 'imdb_rating', 'year']].to_dict('index')

#     all_dfs = []
#     csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv") and f != "movie_metadata.csv"]

#     for fname in csv_files:
#         fpath = os.path.join(folder_path, fname)
#         try:
#             df = pd.read_csv(fpath)
#             movie_key = parse_movie_name(fname)
#             movie_data = financial_lookup.get(movie_key, {})

#             # Map metadata to every row in the review dataframe
#             df["budget"] = clean_currency(movie_data.get("budget"))
#             df["gross"]  = clean_currency(movie_data.get("gross_us"))
#             df["imdb_rating"] = pd.to_numeric(movie_data.get("imdb_rating"), errors='coerce')
#             df["year"] = pd.to_numeric(movie_data.get("year"), errors='coerce')
#             df["movie_title"] = movie_key

#             if "user_rating" in df.columns:
#                 df["user_rating"] = df["user_rating"].apply(parse_user_rating)
            
#             all_dfs.append(df)
#         except Exception as e:
#             print(f"  [skip] Error loading {fname}: {e}")

#     if not all_dfs: raise ValueError("No valid movie review CSVs found.")
#     return pd.concat(all_dfs, ignore_index=True)

# def assign_label(row: pd.Series) -> float:
#     budget, gross = row.get("budget"), row.get("gross")
#     if pd.notna(budget) and budget > 0 and pd.notna(gross):
#         return 1 if (gross / budget) >= 3.0 else 0 # Hit if 3x Budget
#     return np.nan

# # ══════════════════════════════════════════════════════════════════════════════
# # PREPROCESSING
# # ══════════════════════════════════════════════════════════════════════════════

# NUMERIC_FEATURES = ["avg_user_rating", "imdb_rating", "year"]

# def preprocess(df: pd.DataFrame) -> pd.DataFrame:
#     df = df.copy()
    
#     # 1. Assign labels to individual reviews
#     df["label"] = df.apply(assign_label, axis=1)
    
#     # 2. Drop rows with missing financials or text
#     df = df.dropna(subset=["label", "review_text"])
    
#     # 3. Use the individual user rating as the feature
#     if "user_rating" in df.columns:
#         df = df.rename(columns={"user_rating": "avg_user_rating"})
    
#     df["label"] = df["label"].astype(int)
#     return df # Returns 5,000 rows instead of 50

# def build_numeric_matrix(movie_df: pd.DataFrame, scaler: Optional[StandardScaler] = None):
#     X_num = movie_df[NUMERIC_FEATURES].fillna(0).values.astype(np.float32)
#     if scaler is None:
#         scaler = StandardScaler()
#         X_num = scaler.fit_transform(X_num)
#     else:
#         X_num = scaler.transform(X_num)
#     return X_num, scaler, NUMERIC_FEATURES

# # ══════════════════════════════════════════════════════════════════════════════
# # BERT DATASET & MODEL
# # ══════════════════════════════════════════════════════════════════════════════

# class MovieDataset(Dataset):
#     def __init__(self, texts, numeric_features, labels, tokenizer, max_len=256):
#         self.texts = texts
#         self.numeric = torch.tensor(numeric_features, dtype=torch.float32)
#         self.labels = torch.tensor(labels, dtype=torch.float32)
#         self.tok = tokenizer
#         self.max_len = max_len

#     def __len__(self): return len(self.labels)

#     def __getitem__(self, idx):
#         enc = self.tok(self.texts[idx], max_length=self.max_len, 
#                        truncation=True, padding="max_length", return_tensors="pt")
#         return {
#             "input_ids": enc["input_ids"].squeeze(0),
#             "attention_mask": enc["attention_mask"].squeeze(0),
#             "numeric": self.numeric[idx],
#             "label": self.labels[idx]
#         }

# class MovieClassifier(nn.Module):
#     def __init__(self, num_dim: int, dropout: float = 0.3):
#         super().__init__()
#         self.bert = BertModel.from_pretrained("bert-base-uncased")
#         for name, param in self.bert.named_parameters():
#             if not any(f"encoder.layer.{i}" in name for i in [10, 11]):
#                 param.requires_grad = False

#         self.numeric_branch = nn.Sequential(
#             nn.Linear(num_dim, 64), nn.ReLU(), nn.Dropout(dropout)
#         )
#         self.fusion = nn.Sequential(
#             nn.Linear(768 + 64, 256), nn.ReLU(), nn.Dropout(dropout),
#             nn.Linear(256, 1)
#         )

#     def forward(self, input_ids, attention_mask, numeric):
#         bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
#         text_f = bert_out.last_hidden_state[:, 0, :]
#         num_f = self.numeric_branch(numeric)
#         return self.fusion(torch.cat([text_f, num_f], dim=1)).squeeze(1)

# # ══════════════════════════════════════════════════════════════════════════════
# # MAIN EXECUTION
# # ══════════════════════════════════════════════════════════════════════════════

# def train_epoch(model, loader, optimizer, criterion, device):
#     model.train()
#     t_loss, correct, total = 0, 0, 0
#     for b in loader:
#         input_ids, mask, num, labels = b["input_ids"].to(device), b["attention_mask"].to(device), b["numeric"].to(device), b["label"].to(device)
#         optimizer.zero_grad()
#         logits = model(input_ids, mask, num)
#         loss = criterion(logits, labels)
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         optimizer.step()
#         preds = (torch.sigmoid(logits) >= 0.5).float()
#         t_loss += loss.item() * labels.size(0)
#         correct += (preds == labels).sum().item()
#         total += labels.size(0)
#     return t_loss / total, correct / total

# @torch.no_grad()
# def evaluate(model, loader, criterion, device):
#     model.eval()
#     t_loss, correct, total = 0, 0, 0
#     all_p, all_l = [], []
#     for b in loader:
#         input_ids, mask, num, labels = b["input_ids"].to(device), b["attention_mask"].to(device), b["numeric"].to(device), b["label"].to(device)
#         logits = model(input_ids, mask, num)
#         loss = criterion(logits, labels)
#         preds = (torch.sigmoid(logits) >= 0.5).float()
#         t_loss += loss.item() * labels.size(0)
#         correct += (preds == labels).sum().item()
#         total += labels.size(0)
#         all_p.extend(preds.cpu().int().tolist()); all_l.extend(labels.cpu().int().tolist())
#     return t_loss / total, correct / total, all_p, all_l


# if __name__ == "__main__":
#     main('/Users/Diane/Desktop/PSYCH 186B/project/reviews/')

In [ ]:
import os, re, json, pickle, torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress minor warnings for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)

# ══════════════════════════════════════════════════════════════════════════════
# PREPROCESSING HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def clean_curr(v):
    if pd.isna(v) or v == "N/A": return np.nan
    s = re.sub(r'[^\d.]', '', str(v))
    return float(s) if s else np.nan

def parse_m_name(f):
    n = os.path.splitext(f)[0].lower()
    for p in ["imdb", "rt", "rottentomatoes", "reviews"]: n = n.replace(p, "")
    return n.strip("_")

def parse_rating(v):
    if pd.isna(v): return np.nan
    s = str(v).strip()
    if '%' in s: return float(s.replace('%','')) / 10.0
    if '/' in s:
        parts = s.split('/')
        return (float(parts[0])/float(parts[1]))*10.0 if float(parts[1]) != 0 else np.nan
    try: return float(s)
    except: return np.nan

def load_data(path):
    meta = pd.read_csv(os.path.join(path, "movie_metadata.csv"))
    
    # --- ADD THIS LINE TO FIX THE ERROR ---
    meta = meta.drop_duplicates(subset=['movie_key'], keep='last')
    # ---------------------------------------
    
    lookup = meta.set_index('movie_key')[['budget', 'gross_us', 'imdb_rating', 'year']].to_dict('index')
    all_dfs = []
    dropped_count = 0
    
    # The rest of your function remains the same...
    for f in [f for f in os.listdir(path) if f.endswith(".csv") and "metadata" not in f]:
        df = pd.read_csv(os.path.join(path, f))
        key = parse_m_name(f)
        m_data = lookup.get(key, {})
        
        b, g = clean_curr(m_data.get("budget")), clean_curr(m_data.get("gross_us"))
        if pd.isna(b) or pd.isna(g):
            dropped_count += 1
            continue

        df["budget"], df["gross"] = b, g
        df["imdb_rating"] = pd.to_numeric(m_data.get("imdb_rating"), errors='coerce')
        df["year"] = pd.to_numeric(m_data.get("year"), errors='coerce')
        df["movie_title"] = key
        if "user_rating" in df.columns: df["user_rating"] = df["user_rating"].apply(parse_rating)
        all_dfs.append(df)
        
    full_df = pd.concat(all_dfs, ignore_index=True)
    full_df["label"] = full_df.apply(lambda r: 1 if (r['gross']/r['budget']) >= 3.0 else 0, axis=1)
    return full_df.dropna(subset=["review_text"]).rename(columns={"user_rating": "avg_user_rating"}), dropped_count
# ══════════════════════════════════════════════════════════════════════════════
# BERT CLASSES
# ══════════════════════════════════════════════════════════════════════════════

class MovieDataset(Dataset):
    def __init__(self, texts, numeric, labels, tok):
        self.texts, self.numeric, self.labels, self.tok = texts, torch.tensor(numeric, dtype=torch.float32), torch.tensor(labels, dtype=torch.float32), tok
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        e = self.tok(self.texts[i], max_length=256, truncation=True, padding="max_length", return_tensors="pt")
        return {"ids": e["input_ids"].squeeze(0), "mask": e["attention_mask"].squeeze(0), "num": self.numeric[i], "lab": self.labels[i]}

class MovieClassifier(nn.Module):
    def __init__(self, n_dim):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        for n, p in self.bert.named_parameters():
            if "encoder.layer.11" not in n: p.requires_grad = False
        self.num_bnch = nn.Sequential(nn.Linear(n_dim, 64), nn.ReLU())
        self.fusion = nn.Sequential(nn.Linear(768+64, 256), nn.ReLU(), nn.Linear(256, 1))
    def forward(self, ids, mask, num):
        t_f = self.bert(ids, mask).last_hidden_state[:, 0, :]
        n_f = self.num_bnch(num)
        return self.fusion(torch.cat([t_f, n_f], dim=1)).squeeze(1)

# ══════════════════════════════════════════════════════════════════════════════
# MAIN ENGINE
# ══════════════════════════════════════════════════════════════════════════════

def main(path):
    df, dropped = load_data(path)
    
    label_counts = df["label"].value_counts().rename({0: "Flop", 1: "Hit"})
    movie_counts = df.groupby("movie_title")["label"].first().value_counts().rename({0: "Flop", 1: "Hit"})

    print("\n--- GROUND TRUTH SUMMARY ---")
    print(f"Total Movies: {df['movie_title'].nunique()} | Total Reviews: {len(df)}")
    print(f"Movies: {movie_counts.to_dict()}")

    plt.figure(figsize=(6, 4))
    sns.barplot(x=label_counts.index, y=label_counts.values, hue=label_counts.index, legend=False, palette="viridis")
    plt.title("Training Samples (Reviews)"); plt.ylabel("Individual Reviews"); plt.xlabel("Outcome"); plt.show() 

    # Split
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df['movie_title']))
    train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]

    feats = ["avg_user_rating", "imdb_rating", "year"]
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[feats].fillna(0))
    X_test = scaler.transform(test_df[feats].fillna(0))

    # Training
    tok = BertTokenizer.from_pretrained("bert-base-uncased")
    train_ldr = DataLoader(MovieDataset(train_df["review_text"].tolist(), X_train, train_df["label"].tolist(), tok), batch_size=16, shuffle=True)
    test_ldr = DataLoader(MovieDataset(test_df["review_text"].tolist(), X_test, test_df["label"].tolist(), tok), batch_size=16)

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MovieClassifier(len(feats)).to(dev)
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)
    crit = nn.BCEWithLogitsLoss()

    print("\nStarting Training...")
    for ep in range(5):
        model.train()
        t_loss, t_acc, t_count = 0, 0, 0
        for b in train_ldr:
            opt.zero_grad()
            out = model(b["ids"].to(dev), b["mask"].to(dev), b["num"].to(dev))
            loss = crit(out, b["lab"].to(dev))
            loss.backward(); opt.step()
            t_loss += loss.item()
            preds = (torch.sigmoid(out) >= 0.5).float()
            t_acc += (preds == b["lab"].to(dev)).sum().item()
            t_count += b["lab"].size(0)
        print(f"Epoch {ep+1} | Loss: {t_loss/len(train_ldr):.4f} | Acc: {t_acc/t_count:.4f}")

    # Voting
    model.eval(); probs = []
    with torch.no_grad():
        for b in test_ldr:
            probs.extend(torch.sigmoid(model(b["ids"].to(dev), b["mask"].to(dev), b["num"].to(dev))).cpu().numpy())
    
    test_df = test_df.copy(); test_df["p"] = probs
    movie_results = test_df.groupby("movie_title").agg({"label":"first", "p":"mean"})
    movie_results["final_pred"] = (movie_results["p"] >= 0.5).astype(int)
    movie_acc = (movie_results["label"] == movie_results["final_pred"]).mean()

    # ══════════════════════════════════════════════════════════════════════════
    # OUTPUT TO NOTEBOOK (WHAT YOU ASKED FOR)
    # ══════════════════════════════════════════════════════════════════════════
    
    # 1. Prediction Table
    print("\n--- MOVIE PREDICTIONS ---")
    print(movie_results[['label', 'p', 'final_pred']].to_string())

    # 2. Detailed Report
    report = classification_report(movie_results["label"], movie_results["final_pred"], target_names=["Flop", "Hit"])
    print(f"\n--- BERT MOVIE PREDICTION REPORT ---")
    print(f"Total Movies in Test Set: {len(movie_results)}")
    print(f"Movie-Level Accuracy: {movie_acc:.3f}")
    print("\nDetailed Classification Report:")
    print(report)

    # 3. Artifact Summaries
    print("\n--- ARTIFACT SUMMARIES ---")
    print(f"num_cols: {json.dumps(feats)}")
    print(f"scaler (means): {scaler.mean_.tolist()}")
    print(f"model weights: Loaded {len(model.state_dict())} layers into state_dict.")

    # 4. Final Visual
    sns.heatmap(confusion_matrix(movie_results["label"], movie_results["final_pred"]), annot=True, fmt='d', cmap="Blues", 
                xticklabels=["Pred Flop", "Pred Hit"], yticklabels=["Actual Flop", "Actual Hit"])
    plt.title("Confusion Matrix Heatmap (Movies)"); plt.show()

    # Save files quietly in background
    with open("model_weights.pkl", "wb") as f: pickle.dump(model.state_dict(), f)
    with open("scaler.pkl", "wb") as f: pickle.dump(scaler, f)
    with open("num_cols.json", "w") as f: json.dump(feats, f)
    movie_results.to_csv("final_movie_predictions.csv")

if __name__ == "__main__":
    main('/Users/Diane/Desktop/PSYCH 186B/project/reviews/')

ValueError: DataFrame index must be unique for orient='index'.